# Online evals: scoring live spans 🔭

This notebook runs the airline support agent, emits Datadog LLM Observability spans, then
scores those spans with Jev from a separate step and attaches the verdicts as evaluation
metrics.

The two halves stay decoupled. The traced application never imports Jev, and the scorer
never sees a Datadog span ID. They meet on a `turn_id` tag, which means you can score
production traffic asynchronously and backfill history the same way.

Run [1-jev-rubric.ipynb](./1-jev-rubric.ipynb) first if you want the rubric explained.

## Setup

```bash
pip install -r requirements.txt
```

`.env` next to this notebook:

```bash
DD_API_KEY=<your Datadog API key>
DD_APPLICATION_KEY=<your Datadog application key>
DD_SITE=datadoghq.com
OPENAI_API_KEY=<your OpenAI key>
TYPESAFE_API_KEY=<your TypeSafe key>
```

In [ ]:
import json
import os
import time
import uuid

from dotenv import load_dotenv

load_dotenv(override=True)

from ddtrace.llmobs import LLMObs

from jev_rubric import eval_metrics, judge, make_client, verdict
from vega_air import ML_APP, TICKETS, answer_ticket

LLMObs.enable(
    api_key=os.getenv("DD_API_KEY"),
    app_key=os.getenv("DD_APPLICATION_KEY"),
    site=os.getenv("DD_SITE", "datadoghq.com"),
    ml_app=ML_APP,
    agentless_enabled=True,
)

print("ml_app:", ML_APP)

## Phase 1: the application emits spans

Each turn gets a `turn_id` that the application already knows about. Tagging the span with
it is the only coupling the scorer needs. Any stable domain key works: a request ID, a
conversation ID, a message ID.

The turns are also written to a list here. In production that is whatever queue, table, or
topic already holds the traffic you want judged.

In [ ]:
def serve(tickets):
    """The production app. Emits one span per turn, tagged with a turn_id."""
    turns = []
    for ticket in tickets:
        turn_id = f"turn-{uuid.uuid4().hex[:12]}"
        with LLMObs.agent(name="support_turn") as span:
            LLMObs.annotate(
                span=span,
                input_data=ticket["question"],
                tags={"turn_id": turn_id, "ticket_id": ticket["id"]},
            )
            result = answer_ticket(ticket)
            LLMObs.annotate(span=span, output_data=result["reply"])

        turns.append({
            "turn_id": turn_id,
            "ticket_id": ticket["id"],
            "question": ticket["question"],
            "policy_context": result["policy_context"],
            "reply": result["reply"],
            "timestamp_ms": int(time.time() * 1000),
        })
    LLMObs.flush()
    return turns

In [ ]:
turns = serve(TICKETS)

for turn in turns:
    print(f"{turn['ticket_id']}  {turn['turn_id']}  {turn['reply'][:70]}...")

## Phase 2: the scorer

Nothing above imported Jev. This step could run in a different process, on a different
schedule, or over yesterday's traffic.

In [ ]:
client = make_client()

result = judge(client, turns[0]["question"], turns[0]["policy_context"], turns[0]["reply"])
v = verdict(result)

v

## Choosing a metric type

`LLMObs.submit_evaluation` accepts four
[metric types](https://docs.datadoghq.com/llm_observability/instrumentation/sdk/?tab=python):
`score`, `categorical`, `boolean`, and `json`. The choice decides how much of Jev's output
survives into the query layer.

Submit a Noul as a `score` holding the raw probability, and put the pass/fail cut in
`assessment`. Collapsing it to a boolean at submission time destroys the distribution, and
then moving the threshold means re-running the judge over the entire backlog. Keeping the
probability turns a threshold change into a query change.

Use `categorical` for a Choice so the labels can be faceted, and submit its confidence as
a separate `score`. A wrong verdict and an uncertain verdict are different problems, and
you want to be able to tell them apart.

In [ ]:
for metric in eval_metrics(v):
    print(f"{metric['label']:30} {metric['metric_type']:12} {metric['value']}")

Every metric also carries a `judge_model` tag. Aliases move when a release ships, and the
response reports the exact version that answered, so recording it is what lets you compare
two judge models later on the same spans and the same rubric.

In [ ]:
eval_metrics(v)[0]

## Attaching the verdicts to the spans

`span_with_tag_value` is the join. Datadog matches the evaluation to the span carrying that
tag, so the scorer never has to track span IDs.

Pass each turn's own `timestamp_ms` rather than the current time. That makes a re-run
idempotent: scoring a turn again updates its existing verdict instead of leaving a second
one beside it.

In [ ]:
def score(turns):
    """The external evaluator. Judges each turn and writes eval metrics."""
    client = make_client()
    total_tokens = 0

    for turn in turns:
        result = judge(client, turn["question"], turn["policy_context"], turn["reply"])
        v = verdict(result)
        total_tokens += v["input_tokens"]

        for metric in eval_metrics(v):
            LLMObs.submit_evaluation(
                span_with_tag_value={"tag_key": "turn_id", "tag_value": turn["turn_id"]},
                ml_app=ML_APP,
                timestamp_ms=turn["timestamp_ms"],
                **metric,
            )

        print(
            f"  {turn['ticket_id']}  grounded={v['grounded']:.2f} "
            f"impact={v['customer_impact']:.2f} failure={v['failure_mode']} "
            f"(conf {v['failure_mode_confidence']:.2f}) handled={v['handled_correctly']}"
        )

    LLMObs.flush()
    return total_tokens

In [ ]:
total_tokens = score(turns)

print(f"\n{len(turns)} turns x 5 questions, {total_tokens} input tokens")

## What to look at in Datadog

Open **AI Observability > Traces**, filter on the `ml_app` printed at the top of this
notebook, and open one `support_turn` span. The evaluations panel holds all six metrics.

Three views are worth building from here:

- the distribution of `jev_grounded` across spans, with your threshold drawn on it, which
  is the view a boolean metric cannot give you
- `jev_failure_mode` as a facet, to see which failure dominates
- `jev_failure_mode_confidence` over time, because rising uncertainty often means the
  rubric no longer matches the traffic rather than that the agent got worse

Next: **[3-experiments.ipynb](./3-experiments.ipynb)** runs the same rubric over a dataset,
so you can compare two versions of the agent against each other.